In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from researchcodes import (
    define_column_desc, 
    iter_multi_csv_chunks, 
    write_std_h5, 
)

In [2]:
# Download full SDSS Stripe 82 from
# https://faculty.washington.edu/ivezic/sdss/catalogs/stripe82.htm

# Process the text catalog

The RA and Dec errors need calculation, so it's better to process ahead of reading and writing.

In the meantime, we can drop unnecessary columns.

In [6]:
all_column_names = [
    "id_name",
    "ra",
    "dec",
    "ra_rms",
    "dec_rms",
    "Ntot",
    "ISM",
    "SDSS_up_nobs", "SDSS_up_median", "SDSS_up_mean", "SDSS_up_std_err", "SDSS_up_scatter", "SDSS_up_chi^2",
    "SDSS_gp_nobs", "SDSS_gp_median", "SDSS_gp_mean", "SDSS_gp_std_err", "SDSS_gp_scatter", "SDSS_gp_chi^2",
    "SDSS_rp_nobs", "SDSS_rp_median", "SDSS_rp_mean", "SDSS_rp_std_err", "SDSS_rp_scatter", "SDSS_rp_chi^2",
    "SDSS_ip_nobs", "SDSS_ip_median", "SDSS_ip_mean", "SDSS_ip_std_err", "SDSS_ip_scatter", "SDSS_ip_chi^2",
    "SDSS_zp_nobs", "SDSS_zp_median", "SDSS_zp_mean", "SDSS_zp_std_err", "SDSS_zp_scatter", "SDSS_zp_chi^2",
]

raw_text_catalog = pd.read_csv(
    "stripe82calibStars_v4.2.dat",
    sep=r"\s+",
    engine="python",
    header=None,
    names=all_column_names,
    comment="#",      # 直接跳过所有 ### 开头的说明行
)

In [7]:
# calculate the ra ande dec errors
# this formula (rm/sqrt(Ntot)) is 
# in the comments of the catalog file

raw_text_catalog["ra_err"] = raw_text_catalog["ra_rms"]/np.sqrt(raw_text_catalog["Ntot"])

raw_text_catalog["dec_err"] = raw_text_catalog["dec_rms"]/np.sqrt(raw_text_catalog["Ntot"])

In [8]:
# drop the columns we don't want!

process_text_catalog = raw_text_catalog.drop(
    columns=[
        "ra_rms", "dec_rms", 
        "Ntot",
        "ISM",
        "SDSS_up_nobs", "SDSS_up_median", "SDSS_up_scatter", "SDSS_up_chi^2",
        "SDSS_gp_nobs", "SDSS_gp_median", "SDSS_gp_scatter", "SDSS_gp_chi^2",
        "SDSS_rp_nobs", "SDSS_rp_median", "SDSS_rp_scatter", "SDSS_rp_chi^2",
        "SDSS_ip_nobs", "SDSS_ip_median", "SDSS_ip_scatter", "SDSS_ip_chi^2",
        "SDSS_zp_nobs", "SDSS_zp_median", "SDSS_zp_scatter", "SDSS_zp_chi^2",
    ]
)
                      

In [9]:
# save the process catalog text file

process_text_catalog.to_csv(
    "stripe82_processed.dat",
    sep=" ",
    index=False,
)

# Define the HFD5 column description

In [10]:
# Define magnitude column names

# Note this magnitude column order is not necessarily
# to be the same as the column order in the text file. 

# It is OK as long as the filter names in 
# `magnitude_column_names` matches `colnames` defined 
# when reading the text file

magnitude_column_names = [
     "SDSS_up","SDSS_gp","SDSS_rp","SDSS_ip","SDSS_zp",
    "SDSS_up_err","SDSS_gp_err","SDSS_rp_err","SDSS_ip_err","SDSS_zp_err",
]

# define h5 file column description
h5_columns = define_column_desc(
    magnitude_column_names=magnitude_column_names, 
    id_name_length=20, 
)

In [11]:
h5_columns

{'id_name': StringCol(itemsize=20, shape=(), dflt=np.bytes_(b''), pos=0),
 'ra': Float32Col(shape=(), dflt=np.float32(0.0), pos=1),
 'ra_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=2),
 'dec': Float32Col(shape=(), dflt=np.float32(0.0), pos=3),
 'dec_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=4),
 'SDSS_up': Float32Col(shape=(), dflt=np.float32(0.0), pos=5),
 'SDSS_gp': Float32Col(shape=(), dflt=np.float32(0.0), pos=6),
 'SDSS_rp': Float32Col(shape=(), dflt=np.float32(0.0), pos=7),
 'SDSS_ip': Float32Col(shape=(), dflt=np.float32(0.0), pos=8),
 'SDSS_zp': Float32Col(shape=(), dflt=np.float32(0.0), pos=9),
 'SDSS_up_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=10),
 'SDSS_gp_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=11),
 'SDSS_rp_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=12),
 'SDSS_ip_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=13),
 'SDSS_zp_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=14),
 'ipix': Int32Col(shape=(), d

# Read the cvs files

In [12]:
# file path
root_dir = Path(".")
file = root_dir / "stripe82_processed.dat"

In [22]:
# csv column names
colnames = [
    "id_name", "ra", "dec",
    "SDSS_up", "SDSS_up_err",
    "SDSS_gp", "SDSS_gp_err",
    "SDSS_rp", "SDSS_rp_err",
    "SDSS_ip", "SDSS_ip_err",
    "SDSS_zp", "SDSS_zp_err",
    "ra_err", "dec_err",
]

dataframe_iterator = iter_multi_csv_chunks(
    files=file, 
    chunksize=100, 
    read_csv_kwargs={
        "sep": r"\s+", 
        "engine": "python", 
        "header": None, 
        "names": colnames,
        "skiprows": 1, 
    }
)

In [23]:
table_attrs = {
    "ra_unit": "deg",
    "dec_unit": "deg",
    "ra_err_unit": "arcsec",
    "dec_err_unit": "arcsec",
    "version": "v4.2",
    "mag_system": {
        "SDSS_ugriz": "AB",
    },
}

In [24]:
write_std_h5(
    dataframe_iterator=dataframe_iterator, 
    h5_output_path=Path("SDSS_Stripe_82.h5",), 
    group_where="/sdss", 
    group_name="stripe82", 
    group_title="Stripe 82", 
    table_name="std", 
    table_description=h5_columns, 
    table_title="Standard Stars", 
    table_attrs=table_attrs, 
    nside=512, 
    bucket_size=1536 , 
)

Files:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/1 [00:00<?, ?it/s]